<a href="https://colab.research.google.com/github/mohdfazalansari/Automatic-Attendance-Management/blob/main/Data_cleaning_%26_Batch_registration_part.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install MTCNN (for face detection) and a Keras implementation of FaceNet (for face recognition).

In [ ]:
!pip install mtcnn keras-facenet opencv-python

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.4 MB/s eta 0:00:00
  Created wheel for keras-facenet: filename=keras_facenet-0.3.2-py3-none-any.whl size=10367 sha256=7c1265f98589f8643e5c6b51c1acaf3b7b5bea4e2508eb76be7ad5f825a96422
  Stored in directory: /root/.cache/pip/wheels/05/b0/f5/19ac49fedc10b1df3ee56b096edbcfa39d45794fccc6bcdbbf
Successfully built keras-facenet


Data Cleaning

In [ ]:
!unzip "/content/UPLOAD PHOTO (File responses)-20260620T153327Z-3-001.zip" -d "raw_photos"

Archive:  /content/UPLOAD PHOTO (File responses)-20260620T153327Z-3-001.zip
  inflating: raw_photos/UPLOAD PHOTO (File responses)/Devsharma - DEV SHARMA.jpeg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/IMG-20260617-WA0006 - ARYAN PATIDAR.jpg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/IMG-20260509-WA0007 - PRIYANSHI SEN.jpg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/2026-02-21-12-36-22-977 - NAMAN GAUR.jpg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/IMG-20221225-WA0004 - ALAQMAR KANCHWALA.jpg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/WhatsApp Image 2026-06-15 at 8.44.25 AM - ZAINAB ABBAS.jpeg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/78059177-5746-47c8-a8bb-a45a16a46039 - Avdhesh Thakur.jpeg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/picture - GAURAVI GUPTA.jpeg  
  inflating: raw_photos/UPLOAD PHOTO (File responses)/Snapchat-1607742645~2 - ANSHITA KUMBHARE.jpg  
  inflating: raw_photos/UPLOAD PHO

In [ ]:
import re
import shutil
from pathlib import Path

SOURCE_FOLDER = Path("/content/raw_photos/UPLOAD PHOTO (File responses)")
DEST_FOLDER = Path("/content/labeled_dataset_new")

# Create destination folder if it doesn't exist
DEST_FOLDER.mkdir(parents=True, exist_ok=True)

print("Cleaning dataset...\n")

# Initialize summary counters
processed = 0
duplicates = 0
skipped = 0
corrupted = 0

# Check if source directory exists before looping
if not SOURCE_FOLDER.exists():
    print(f"Error: Source folder not found at {SOURCE_FOLDER}")
else:
    # Iterate using Pathlib
    for file_path in SOURCE_FOLDER.iterdir():

        # Skip if it's a directory
        if not file_path.is_file():
            continue

        # Keep original extension
        ext = file_path.suffix.lower()
        if ext not in [".jpg", ".jpeg", ".png"]:
            skipped += 1
            continue

        try:
            base_name = file_path.stem

            # -----------------------------
            # CASE 1: Google Forms pattern
            # -----------------------------
            if "__" in base_name:
                clean_name = base_name.split("__")[-1]
            # -----------------------------
            # CASE 2: Old pattern
            # -----------------------------
            elif "-" in base_name:
                clean_name = base_name.split("-", 1)[1]
            else:
                clean_name = base_name

            # Replace separators with spaces
            clean_name = clean_name.replace("_", " ")

            # Remove anything except letters and spaces
            clean_name = re.sub(r"[^A-Za-z ]", "", clean_name)

            # Remove extra spaces and strip padding
            clean_name = re.sub(r"\s+", " ", clean_name).strip()

            # If name is entirely wiped out by regex, skip it
            if not clean_name:
                print(f"Skipped (No valid name extracted): {file_path.name}")
                skipped += 1
                continue

            # Convert to UPPERCASE and swap spaces for underscores
            clean_name = clean_name.upper().replace(" ", "_")

            # Final filename (Preserving the original extension!)
            final_name = f"{clean_name}{ext}"
            destination_path = DEST_FOLDER / final_name

            # Handle duplicates
            is_duplicate = False
            counter = 1
            while destination_path.exists():
                is_duplicate = True
                # Format: NAME_1.jpg, NAME_2.jpg
                destination_path = DEST_FOLDER / f"{clean_name}_{counter}{ext}"
                counter += 1

            if is_duplicate:
                duplicates += 1

            # Copy file to new destination
            shutil.copy2(file_path, destination_path)
            processed += 1

            print(f"✓ {file_path.name} -> {destination_path.name}")

        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")
            corrupted += 1

# ==============================
# Summary
# ==============================
print("\n" + "=" * 50)
print("DATASET CLEANING COMPLETE")
print("=" * 50)
print(f"Processed  : {processed}")
print(f"Duplicates : {duplicates}")
print(f"Skipped    : {skipped}")
print(f"Corrupted  : {corrupted}")
print(f"Output     : {DEST_FOLDER}")

Cleaning dataset...

✓ IMG-20260612-WA0001 - MISHIKA KASLIWAL.jpg -> WA_MISHIKA_KASLIWAL.jpg
✓ IMG_0432 - MAHAK AGRAWAL.jpeg -> MAHAK_AGRAWAL.jpeg
✓ IMG-20251108-WA0026 - ANUSHKA SHRIVASTAV.jpg -> WA_ANUSHKA_SHRIVASTAV.jpg
✓ PHOTO-2026-06-13-18-27-03 - RABLEEN KAUR RANDHAWA.png -> RABLEEN_KAUR_RANDHAWA.png
✓ file_00000000b33c71fa83c755612a12031a - D.L Akash.png -> DL_AKASH.png
✓ 20260507_132711 - NANDINI SHAW.jpg -> NANDINI_SHAW.jpg
✓ IMG-20260613-WA0088 - SHUBHANSHU SAHU.jpg -> WA_SHUBHANSHU_SAHU.jpg
✓ IMG_20260605_203254 - KAPIL YADUWANSHI.jpg -> KAPIL_YADUWANSHI.jpg
✓ 1778936301125 - ARYA JAIN.png -> ARYA_JAIN.png
✓ profile - NANDINI SINGH.jpg -> NANDINI_SINGH.jpg
✓ imagmeee - SALONI DUBEY.jpg -> SALONI_DUBEY.jpg
✓ IMG_20260614_002459 - PALKESH GUPTA.jpg -> PALKESH_GUPTA.jpg
✓ IMG-20260522-WA0000 - GAUTAM CHOUDHARY.jpg -> WA_GAUTAM_CHOUDHARY.jpg
✓ WhatsApp Image 2025-07-17 at 23.22.04_ee5528c5 - KHWAHISH CHOUKSEY.jpg -> AT_EEC_KHWAHISH_CHOUKSEY.jpg
✓ IMG-20260527-WA0105 - DISHA VISH

In [ ]:
#Download Labled dataset
import shutil
from google.colab import files
import os

# Define the paths
folder_path = '/content/labeled_dataset_new'
zip_path = '/content/labeled_dataset_new.zip'

print("Compressing folder, please wait...")

# Create the zip file
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', folder_path)

print(f"Folder zipped successfully. Size: {os.path.getsize(zip_path) / (1024 * 1024):.2f} MB")
print("Starting download...")

# Trigger the download
files.download(zip_path)

Compressing folder, please wait...
Folder zipped successfully. Size: 117.39 MB
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Batch Database Registration

In [ ]:
!pip install tensorflow
import os
import cv2
import numpy as np
import sqlite3
import json
from mtcnn import MTCNN
from keras_facenet import FaceNet

print("Loading Models...")
detector = MTCNN()
embedder = FaceNet()

# Setup SQLite Database
conn = sqlite3.connect('attendance_system.db')
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE,
        embedding TEXT
    )
''')
conn.commit()

dataset_path = "/content/labeled_dataset_new"
print(f"\nScanning '{dataset_path}' for student photos...\n" + "-"*40)

for filename in os.listdir(dataset_path):
    # Only process image files
    if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    # The filename is now clean (e.g., "John_Doe.jpg")
    # We remove the extension and replace underscores back to spaces for the database
    student_name = os.path.splitext(filename)[0].replace("_", " ")
    image_path = os.path.join(dataset_path, filename)

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not read {filename}")
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Detect face
    results = detector.detect_faces(img_rgb)
    if not results:
        print(f"Skipping {student_name}: No face detected.")
        continue

    # Get the bounding box of the most prominent face
    x, y, w, h = results[0]['box']
    x, y = abs(x), abs(y)

    # Crop the face
    face_crop = img_rgb[y:y+h, x:x+w]
    face_crop = cv2.resize(face_crop, (160, 160))

    # Extract embedding
    face_crop = np.expand_dims(face_crop, axis=0)
    embedding = embedder.embeddings(face_crop)[0]

    # Save to Database
    try:
        embedding_json = json.dumps(embedding.tolist())
        cursor.execute("INSERT INTO students (name, embedding) VALUES (?, ?)", (student_name, embedding_json))
        conn.commit()
        print(f"Success: Registered '{student_name}'")
    except sqlite3.IntegrityError:
        print(f"Skipped: '{student_name}' is already in the database.")

print("-" * 40 + "\nBatch registration complete!")

Loading Models...

Scanning '/content/labeled_dataset_new' for student photos...
----------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Success: Registered 'DIVYANSHI PRAJAPATI'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
Success: Registered 'MSKRITIKA GEEL'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
Success: Registered 'KASHIFA SHAH'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
Success: Registered 'DIVYANSHU JAISWAL'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Success: Registered 'MAHAK RATHOD'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
Success: Registered 'SHRIDHI JAIN'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Success: Registered 'KUSHAGRA BAKLIWAL'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Success: Registered 'ANSHITA SONI'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
Success: Registered 'AMAN SHARMA'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step
Success: Registered 'VEDANSHI GUPTA'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step
Success: Registered 'AADARSH SAHU'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step
Success:

The Video Inference Script for testing

In [ ]:
import cv2
import numpy as np
import sqlite3
import json
from mtcnn import MTCNN
from keras_facenet import FaceNet
from scipy.spatial.distance import cosine

print("Loading Models...")
detector = MTCNN()
embedder = FaceNet()

# 1. Load Known Students from Database
print("Loading database...")
conn = sqlite3.connect('attendance_system.db')
cursor = conn.cursor()
cursor.execute("SELECT name, embedding FROM students")
known_students = []
for row in cursor.fetchall():
    name = row[0]
    embedding = np.array(json.loads(row[1]))
    known_students.append((name, embedding))
conn.close()

# 2. Setup Video Ingestion and Output
input_video_path = '/content/test_video.mp4' # Change this if your file has a different name
output_video_path = 'output_recognition.mp4'

cap = cv2.VideoCapture(input_video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Create VideoWriter to save the output in Colab
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print("Processing Video... This may take a few minutes depending on the video length.")

frame_count = 0
skip_frames = 2 # Process every 2nd frame to speed up Colab processing

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Process only every Nth frame
    if frame_count % skip_frames == 0:
        # Convert to RGB for MTCNN
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Detect faces
        results = detector.detect_faces(img_rgb)

        for result in results:
            x, y, w, h = result['box']
            x, y = abs(x), abs(y)

            # Extract and format the face crop
            face_crop = img_rgb[y:y+h, x:x+w]
            if face_crop.shape[0] == 0 or face_crop.shape[1] == 0:
                continue

            face_crop = cv2.resize(face_crop, (160, 160))
            face_crop = np.expand_dims(face_crop, axis=0)

            # Get live embedding
            live_embedding = embedder.embeddings(face_crop)[0]

            # Compare with database using Cosine Similarity
            best_match_name = "Unknown"
            lowest_distance = 1.0 # Set a default high distance
            threshold = 0.5 # If distance is lower than this, it's a match

            for name, db_embedding in known_students:
                distance = cosine(live_embedding, db_embedding)
                if distance < lowest_distance and distance < threshold:
                    lowest_distance = distance
                    best_match_name = name

            # Draw Bounding Box and Name on the frame
            color = (0, 255, 0) if best_match_name != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
            cv2.putText(frame, best_match_name, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    # Write the frame (processed or skipped) to the output video
    out.write(frame)

cap.release()
out.release()
print(f"Complete! Download '{output_video_path}' from the left sidebar to see the results.")

Loading Models...
Loading database...
Processing Video... This may take a few minutes depending on the video length.
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━

Using MTCNN in browser testing using webcame.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
from mtcnn import MTCNN
from keras_facenet import FaceNet
from scipy.spatial.distance import cosine
import sqlite3
import json

# 1. Initialize Models & Load Database
print("Loading Models & Database...")
detector = MTCNN()
embedder = FaceNet()

conn = sqlite3.connect('attendance_system.db')
cursor = conn.cursor()
cursor.execute("SELECT name, embedding FROM students")
known_students = [(row[0], np.array(json.loads(row[1]))) for row in cursor.fetchall()]
conn.close()

# 2. JavaScript to Access Browser Camera
def take_photo(quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Create a capture button
      const btn = document.createElement('button');
      btn.textContent = 'Capture Frame for Recognition';
      div.appendChild(btn);

      await new Promise((resolve) => btn.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  image = np.asarray(bytearray(binary), dtype="uint8")
  return cv2.imdecode(image, cv2.IMREAD_COLOR)

# 3. Capture and Process
try:
  print("Starting webcam... Click 'Capture Frame' when ready.")
  frame = take_photo()

  # Process the captured frame
  img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  results = detector.detect_faces(img_rgb)

  if not results:
      print("No face detected in the captured frame.")
  else:
      for result in results:
          x, y, w, h = result['box']
          x, y = abs(x), abs(y)

          face_crop = img_rgb[y:y+h, x:x+w]
          face_crop = cv2.resize(face_crop, (160, 160))
          face_crop = np.expand_dims(face_crop, axis=0)

          live_embedding = embedder.embeddings(face_crop)[0]

          best_match_name = "Unknown"
          lowest_distance = 1.0

          # UPDATE YOUR THRESHOLD HERE TO FIX WRONG PREDICTIONS
          strict_threshold = 0.40

          for name, db_embedding in known_students:
              distance = cosine(live_embedding, db_embedding)
              if distance < lowest_distance and distance < strict_threshold:
                  lowest_distance = distance
                  best_match_name = name

          print(f"Prediction: {best_match_name} (Distance: {lowest_distance:.3f})")

          # Draw bounding box and show image
          color = (0, 255, 0) if best_match_name != "Unknown" else (0, 0, 255)
          cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
          cv2.putText(frame, f"{best_match_name} ({lowest_distance:.2f})", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

      from google.colab.patches import cv2_imshow
      cv2_imshow(frame)

except Exception as err:
  print(str(err))

Loading Models & Database...
Starting webcam... Click 'Capture Frame' when ready.


<IPython.core.display.Javascript object>

NotAllowedError: Permission denied
